# Random Forest desde cero: demostración y experimentos

Este notebook reutiliza los módulos del proyecto y solo procesa telemetría de desarrollo. `Untitled.ipynb` es la referencia histórica, no es necesario ejecutarlo.

La validación interna usa los mismos incidentes en todas las corridas si no se cambia su tamaño ni la semilla de partición. El detector usa calibración previa de desarrollo: esta evaluación es exploratoria, no una estimación independiente. La reserva final no interviene.

In [ ]:
import sys
from pathlib import Path

from IPython.display import display

# Resolver imports tanto desde la raíz como desde notebooks/.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.forest_experiment import demo_roundtrip, load_development, run_experiment

# Usar None si los datos están en la ruta de config/topology.json.
DATA_ROOT = project_root / "data/RCAEval"
VALIDATION_INCIDENTS = 12
SPLIT_SEED = 42
FOREST_SEED = 42
# Cada pareja indica (incidentes completos de entrenamiento, árboles).
RUNS = [(12, 10), (24, 10)]
PLOT = True

In [ ]:
# Ejecutar detección y asociación una sola vez; no es parte del tiempo de ajuste.
development = load_development(project_root, data_root=DATA_ROOT)
print(f"Preparación de desarrollo: {development.elapsed_sec:.3f} s")

## Tiempo de ejecución

El ajuste incluye el bootstrap y la construcción concurrente; no incluye detección, asociación ni selección de columnas. El tiempo de evaluación incluye predicción y ranking. Son mediciones de una corrida local, no un benchmark controlado. Los tamaños solicitados incluyen incidentes omitidos por el detector; se informa cuántos aportaron filas.

In [ ]:
for train_size, tree_count in RUNS:
    result = run_experiment(
        development, train_incidents=train_size, validation_incidents=VALIDATION_INCIDENTS,
        split_seed=SPLIT_SEED, n_trees=tree_count, forest_seed=FOREST_SEED,
    )
    associated = result.train_audit.status.eq("associated").sum()
    print(f"Train: {associated}/{train_size} incidentes asociados, {len(result.X_train)} filas, "
          f"{result.X_train.shape[1]} variables, {tree_count} árboles")
    print(f"Ajuste: {result.fit_seconds:.3f} s; evaluación: {result.evaluation_seconds:.3f} s")

## Calidad exploratoria de la última corrida

Se seleccionan columnas finitas y variables solo en entrenamiento; no hay imputación. Si a un candidato le falta una variable seleccionada, se omite el incidente completo del ranking, pero se cuenta en la cobertura. Top-1, Top-3 y MRR se calculan solo sobre incidentes puntuados. Los empates se resuelven por nombre de servicio.

In [ ]:
display(result.validation.summary.to_frame("Valor"), result.validation.incidents)
if PLOT:
    import matplotlib.pyplot as plt
    from src.forest_plots import plot_validation

    plot_validation(result.validation)
    plt.show()

In [ ]:
# Mostrar las variables asignadas a cada árbol y tres predicciones tras recargar el JSON.
# Son filas de entrenamiento para demostrar la interfaz, no métricas de calidad.
print(result.forest.training_features)
display(demo_roundtrip(result.forest, result.X_train.head(3)))